# What Makes a Good Negative?

Team 16 -- Farhan Sadaf, Argha Dhar, Jacob Matti, and Aditi Ganapathi.

We compare ways of building negative TCR-peptide pairs.

Data sources:

- Positive pairs: [IMMREP25](https://www.kaggle.com/competitions/immrep25/data), downloaded from this [public mirror](https://github.com/Ajaingithub/pMHC_TCR_specificity/tree/8ebcab1d856eaebc21297c17249e1f6860aea263/data): IEDB (9,652 rows) and VDJdb (2025-02-21 release, 30,274 rows).
- Background TCRs: [10x healthy-donor PBMC dataset](https://www.10xgenomics.com/datasets/human-pbmc-from-a-healthy-donor-10-k-cells-v-2-2-standard-5-0-0), Cell Ranger 5.0.0. We download its matching amino-acid annotations when needed.
- MHC sequences: [IPD-IMGT/HLA 3.66.0](https://github.com/ANHIG/IMGTHLA/tree/3660), using extracellular regions for A*02:01 and B*40:01.
- Beta2-microglobulin: [UniProt P61769](https://www.uniprot.org/uniprotkb/P61769/entry), using the mature protein.


## Setup

Select **Python (CSE 559 project)**. We use five seeds and the five saved peptide folds.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split

from preprocessing.prepare import prepare_data
from preprocessing.sequences import prepare_receptors
from evaluation import evaluate as evaluation
from evaluation.comparison import prepare_common_benchmark, compare_predictors, compare_after_audit
from evaluation.predictors import reproduce_originals, predict_common_pairs
from hard_negatives import experiment as negatives, structures

if Path(sys.prefix).resolve() != Path('.venv').resolve():
    raise RuntimeError('Select Python (CSE 559 project) from the project folder.')

N_JOBS = 3
SEEDS = [13, 29, 47, 71, 101]
RATIO = 5
SHUFFLES = 1000
BOOTSTRAPS = 1000
EPOCHS = 10
DEVICE = 'cpu'

results = Path('results')
supporting = results / 'supporting'
figures = results / 'figures'
supporting.mkdir(parents=True, exist_ok=True)
figures.mkdir(exist_ok=True)

## Preprocessing

We clean the pairs, keep the two HLA alleles, and reuse the peptide folds.

[Guide](preprocessing/GUIDE.md)

In [ ]:
positives, repertoire, coverage = prepare_data('data', cdr3_window=(8, 30))
positives.to_csv(supporting / 'positives.csv', index=False)
repertoire.to_csv(supporting / 'repertoire.csv', index=False)
coverage.to_csv(supporting / 'fold_coverage.csv')
receptors = prepare_receptors(output=supporting / 'receptor_sequences.csv')
display(coverage)

## Evaluation pairs

We reserve 20% of the background TCRs. Every method uses the same length-matched test pairs.

[Guide](evaluation/GUIDE.md)

In [ ]:
train_background, test_background = train_test_split(repertoire, test_size=0.2, random_state=1601)
assert not set(train_background.tcr_beta) & set(test_background.tcr_beta)

test_sets = {}
for fold in sorted(positives.fold.unique()):
    heldout = positives.loc[positives.fold.eq(fold)]
    test_sets[fold] = evaluation.make_test_set(heldout, test_background, seed=1601 + fold, ratio=RATIO)

pd.concat([table.assign(fold=fold) for fold, table in test_sets.items()], ignore_index=True).to_csv(
    supporting / 'test_pairs.csv.gz', index=False,
)
common_pairs = prepare_common_benchmark(supporting, receptors=receptors)

## Shortcut controls

We use frozen ESM-2 vectors and run all seven input configurations on all seven negative sets.

In [ ]:
sequences = set(positives.tcr_beta) | set(positives.peptide) | set(repertoire.tcr_beta)
embeddings = negatives.embed_sequences(sequences, device=DEVICE, cache=supporting / 'embeddings.npz')
control_summary = evaluation.run_controls(
    positives, train_background, test_sets, embeddings, SEEDS,
    ratio=RATIO, epochs=EPOCHS, permutations=SHUFFLES, bootstraps=BOOTSTRAPS,
    n_jobs=N_JOBS, output=supporting,
)

## Negative construction

We compare random pairing, background sampling, and p99 through p1. We also run the 3-mer boosting check.

[Guide](hard_negatives/GUIDE.md)

In [ ]:
negative_summary = negatives.run_experiments(
    positives, train_background, test_sets, embeddings, SEEDS,
    ratio=RATIO, epochs=EPOCHS, permutations=SHUFFLES, bootstraps=BOOTSTRAPS,
    n_jobs=N_JOBS, output=supporting,
)
columns = ['method', 'auroc_mean', 'auroc_std', 'auprc_mean', 'auprc_std']
main_results = negative_summary.loc[negative_summary.experiment.eq('negative_swap'), columns]
display(main_results)

## Figures

Error bars show standard deviations across seed averages. Detailed metrics stay in the supporting folder.

In [ ]:
negatives.plot_hardness(supporting)
negatives.plot_robustness(supporting)
swap_results = negatives.plot_negative_swap(supporting)
random_controls = evaluation.plot_controls(supporting, method='random_pairing')
hard_controls = evaluation.plot_controls(supporting, method='p1')

for image in supporting.glob('*.png'):
    image.replace(figures / image.name)

## Published predictors

We show the reported author-test scores and repeat those runs when their setup is available. Run logs go to the supporting folder.

[Guide](tcr_predictors/GUIDE.md)

In [ ]:
display(Markdown(Path('tcr_predictors/RESULTS.md').read_text()))
reproduce_originals(supporting)

## Shared benchmark

Both released models score the same supported pairs. We compare their gap with the local negative-swap range on those pairs.

In [ ]:
common_scores = predict_common_pairs(supporting)
predictor_metrics, rq5_comparison = compare_predictors(common_scores, folder=supporting)
display(predictor_metrics)
display(rq5_comparison)

## Structural audit

We fold 200 hard pairs, apply the fixed confidence rule, and repeat p1 after excluding flagged pairs. Confidence does not confirm binding.

In [ ]:
STRUCTURE_THRESHOLD = 0.6
structures.prepare_sequences(supporting, receptors=receptors)
structures.prepare_inputs(
    supporting / 'structure_candidates.csv', supporting / 'structure_sequences.csv', supporting / 'boltz_inputs',
)
structures.fold_pairs(inputs=supporting / 'boltz_inputs', output=supporting / 'boltz')
audit = structures.score_structures(
    threshold=STRUCTURE_THRESHOLD, inputs=supporting / 'structure_inputs.csv',
    folder=supporting / 'boltz', output=supporting / 'structure_scores.csv',
)

blocked = audit.loc[audit.plausible_binder.eq(1), ['hla', 'peptide', 'tcr_beta']]
filtered_summary = negatives.run_experiments(
    positives, train_background, test_sets, embeddings, SEEDS,
    ratio=RATIO, epochs=EPOCHS, permutations=SHUFFLES, bootstraps=BOOTSTRAPS,
    n_jobs=N_JOBS, methods=['p1'], excluded_pairs=blocked, output=supporting / 'after_audit',
)
audit_change = compare_after_audit(supporting)
display(audit_change)

## Results

We save a short summary in `results/RESULTS.md`. Predictions, caches, and detailed tables stay in `results/supporting/`.

In [ ]:
tables = {
    'Negative construction': main_results,
    'Shared benchmark': predictor_metrics.rename_axis('predictor').reset_index(),
    'Negative-swap comparison': rq5_comparison.iloc[:, :4],
    'Change after structural filtering': audit_change.rename_axis('statistic').reset_index(),
}
report = [
    '# Results\n',
    f'{len(positives)} positive pairs, {positives.peptide.nunique()} peptides, five folds, '
    f'{len(SEEDS)} seeds, {EPOCHS} epochs, and {RATIO} negatives per positive.\n',
    f'{SHUFFLES} label shuffles and {BOOTSTRAPS} peptide bootstraps. '
    'Main scores are averaged over folds, then seeds. AP is average precision.\n',
]
for title, table in tables.items():
    report += [f'\n## {title}\n', '| ' + ' | '.join(table.columns) + ' |', '|' + '---|' * len(table.columns)]
    for row in table.itertuples(index=False, name=None):
        values = [f'{value:.4f}' if isinstance(value, float) else str(value) for value in row]
        report.append('| ' + ' | '.join(values) + ' |')

report += [
    f'\nStructural audit: {len(audit)} pairs; {int(audit.plausible_binder.sum())} flagged '
    f'at confidence threshold {STRUCTURE_THRESHOLD}.\n',
    'Negatives are assumed and the background is one donor. Some TCRs cross folds; '
    'one fold contains a single peptide. Confidence flags do not confirm binding. '
    'Pretrained training overlap is not ruled out.\n',
    '[Figures](figures/) · [Supporting files](supporting/)',
]
(results / 'RESULTS.md').write_text('\n'.join(report));